# 10 · ส่งออกค่าสถิติสำหรับเว็บนำเสนอ

เว็บใน `Website_for_presentation/` วาดกราฟจากไฟล์ `aggregates.json` ที่เซลล์นี้เขียน
ไฟล์มีเฉพาะค่าสรุปรวม (สหสัมพันธ์ สัมประสิทธิ์และเมทริกซ์ความแปรปรวนร่วมของสมการถดถอย และจำนวนคนต่อช่วงชั่วโมง)
ไม่มีแถวของผู้ตอบคนใด จึง commit ได้ในขณะที่ `data/` อยู่นอก git

ถ้ารันนอก repo (เช่นบน Colab) จะไม่มีโฟลเดอร์เว็บ เซลล์นี้จะคำนวณแต่ไม่เขียนไฟล์

In [1]:
import os
import pathlib
import sys

# notebook.nvim starts the kernel in nvim's cwd; step into codingpy/ so common.py
# and data/ are found when the notebook is opened from the repo root
if pathlib.Path("codingpy/common.py").is_file():
    os.chdir("codingpy")
    sys.path.insert(0, os.getcwd())

%matplotlib inline

In [2]:
import json

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

from common import CONTROLS, DIMENSIONS, PREDICTORS, formula, load_data, passed_attention

survey = load_data()  # every response, cleaned
df = passed_attention(survey)  # the sample every research question uses
print(f"{len(survey)} responses -> {len(df)} passed the attention check")

121 responses -> 107 passed the attention check


In [3]:
OUT_PATH = pathlib.Path("../Website_for_presentation/src/data/generated/aggregates.json")


def fisher_ci(r: float, n: int) -> tuple[float, float]:
    """95% CI of Pearson r through the Fisher z transform."""
    z = np.arctanh(r)
    half = stats.norm.ppf(0.975) / np.sqrt(n - 3)
    return float(np.tanh(z - half)), float(np.tanh(z + half))


def correlations(df: pd.DataFrame) -> list[dict]:
    """Pearson r of every predictor with every dimension, pairwise complete."""
    rows = []
    for x in PREDICTORS:
        for col, name in DIMENSIONS.items():
            pair = df[[x, col]].dropna()
            r, p = stats.pearsonr(pair[x], pair[col])
            low, high = fisher_ci(r, len(pair))
            rows.append(
                {
                    "predictor": x,
                    "dimension": name.lower(),
                    "r": round(float(r), 4),
                    "p": round(float(p), 4),
                    "ciLow": round(low, 4),
                    "ciHigh": round(high, 4),
                    "n": len(pair),
                }
            )
    return rows


def regression(df: pd.DataFrame) -> dict:
    """Block 1 (controls) and block 2 (all eight) per dimension, HC3 errors as in chapter 4."""
    data = df[PREDICTORS + list(DIMENSIONS)].dropna().reset_index(drop=True)
    models = {}
    for col, name in DIMENSIONS.items():
        reduced = smf.ols(formula(col, CONTROLS), data=data).fit()
        full = smf.ols(formula(col, PREDICTORS), data=data).fit(cov_type="HC3")
        ci = full.conf_int()
        # HC3 covariance of [Intercept, *PREDICTORS]; the site turns it into the CI of a prediction
        cov = full.cov_params().loc[["Intercept", *PREDICTORS], ["Intercept", *PREDICTORS]]
        models[name.lower()] = {
            "intercept": round(float(full.params["Intercept"]), 4),
            "r2Block1": round(float(reduced.rsquared), 4),
            "r2": round(float(full.rsquared), 4),
            "coefficients": [
                {
                    "predictor": x,
                    "b": round(float(full.params[x]), 4),
                    "se": round(float(full.bse[x]), 4),
                    "ciLow": round(float(ci.loc[x, 0]), 4),
                    "ciHigh": round(float(ci.loc[x, 1]), 4),
                    "p": round(float(full.pvalues[x]), 4),
                }
                for x in PREDICTORS
            ],
            "covariance": [[round(float(v), 8) for v in row] for row in cov.to_numpy()],
        }
    ranges = [
        {
            "predictor": x,
            "mean": round(float(data[x].mean()), 4),
            "min": float(data[x].min()),
            "max": float(data[x].max()),
        }
        for x in PREDICTORS
    ]
    return {"n": len(data), "predictors": ranges, "models": models}


def hours_histogram(df: pd.DataFrame) -> list[dict]:
    """Respondents per one-hour band of daily social media use."""
    hours = df["social_media_hours"].dropna()
    edges = np.arange(np.floor(hours.min()), np.ceil(hours.max()) + 1)
    counts, _ = np.histogram(hours, bins=edges)
    return [
        {"from": float(a), "to": float(b), "count": int(c)}
        for a, b, c in zip(edges[:-1], edges[1:], counts)
    ]


def last_digit_share(df: pd.DataFrame) -> dict:
    """Share of hours answers by first decimal digit; people favour .0 and .5."""
    hours = df["social_media_hours"].dropna()
    digit = (np.round(hours * 10) % 10).astype(int)
    share = digit.value_counts(normalize=True).reindex(range(10), fill_value=0.0)
    return {str(d): round(float(s), 4) for d, s in share.items()}


aggregates = {
    "source": "codingpy/10_export_website.ipynb",
    "sample": {"responses": len(survey), "passed": len(df)},
    "correlations": {"passed": correlations(df), "all": correlations(survey)},
    "regression": regression(df),
    "hoursHistogram": hours_histogram(df),
    "hoursDecimalDigit": last_digit_share(df),
}

# the site only exists inside the repo; anywhere else there is nothing to update
if OUT_PATH.parents[3].is_dir():
    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    OUT_PATH.write_text(json.dumps(aggregates, ensure_ascii=False, indent=2) + "\n")
    print(f"saved: {OUT_PATH}")
else:
    print(f"{OUT_PATH.parents[3]} not found; aggregates computed but not written")

saved: ../Website_for_presentation/src/data/generated/aggregates.json
